# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/00Bhuwan/flyrank-ml-internship-my-work/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Scoring / Ranking.** The decision is: given a finite review queue, which pages go to the top? We need a priority score per page so editors review the highest-scoring ones first. This maps to a scoring task (produce a calibrated priority score) that directly enables ranking by that score. Classification (is_declining_label) is the training target; scoring is the operational output.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. Target or proxy

**Target:** `is_declining_label` (1 when `trend_direction == "down"`, 0 otherwise) — a proxy derived from the *current* 90-day window comparing last-30d vs prev-30d impressions.  
**Source:** Observed outcome in the data pipeline (`trend_direction` computed from `trend_pct`), not a human-defined rule.  
**Important:** This is a *current-window* decline proxy, not a forward-looking label. A stronger version would use prior-90d features → next-30d decline/recovery. For now we frame honestly: we score pages by how much they look like they're declining *now*, to prioritize review.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 3. Success metric

**Precision@K (K = review capacity per cycle, e.g., 50 or 100 pages).**  
Why: The decision is a capacity-constrained queue. Editors review the top-K scored pages. What matters is how many of those K are truly declining (worth reviewing).  
**Good number:** Precision@100 ≥ 0.60 on a client-holdout test split (i.e., ≥ 60 of the top 100 scored pages are actually declining). This beats the baseline rule (stale + visible) which achieves ~0.10 precision@100 on the same split (see w07 baseline).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 4. The unit of analysis, as a real dataframe

One row = one content page (`content_id`), aggregated over its available trailing 90-day window of search/engagement signals. 30,000 rows × 44 columns in the starter slice.

In [4]:
import pandas as pd

csv_path = r"C:\\Users\\9849i\\OneDrive\\Documents\\Bhuwan\\Python Scripts\\flyrank-ml-internship-my-work\\data\\raw\\content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nFirst 3 rows:")
print(df.head(3).to_string())
print("\nLabel distribution:")
print(df['trend_direction'].value_counts())
print("\nClient count:", df['client_id'].nunique())
print("\nContent types:", df['content_type'].value_counts().to_dict())
print("\nMissingness sample:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_dict())

Shape: (30000, 44)

Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

First 3 rows:
             content_id          client_id  search_volume  competition competition_level   cpc     content_type    main_intent  word_count  char_count provider_used              model_u

## 5. Why ML beats a fixed rule here

A fixed rule like "days_since_last_update ≥ 180 AND impressions_90d ≥ 500" catches only 0.057% of pages (see w01: 17/30,000). The pattern of decline is **multivariate and context-dependent**: a page with moderate impressions but dropping CTR *and* rising scroll_rate *and* stale content *and* informational intent may be a better refresh candidate than a high-impression page with stable CTR. The signals (impressions, position, CTR, engagement, age, freshness, word count, intent, content_type) interact in ways that differ by client and content type — too many conditional branches for an if-statement. ML learns the weighting from data; a rule hardcodes one person's guess.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Quick verification: the fixed rule catches very few
import pandas as pd
csv_path = r"C:\\Users\\9849i\\OneDrive\\Documents\\Bhuwan\\Python Scripts\\flyrank-ml-internship-my-work\\data\\raw\\content_refresh_anonymized.csv"
df = pd.read_csv(csv_path)
fixed_rule = (df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)
print(f"Fixed rule catches: {fixed_rule.sum()} / {len(df)} = {fixed_rule.mean():.4%}")
print(f"Of those, declining: {(fixed_rule & (df['trend_direction'] == 'down')).sum()} / {fixed_rule.sum()}")

# Show multivariate signal diversity
print("\nCorrelation of numeric features with is_declining:")
numeric_cols = df.select_dtypes(include='number').columns
corrs = df[numeric_cols].corrwith((df['trend_direction'] == 'down').astype(int)).abs().sort_values(ascending=False)
print(corrs.head(10))

Fixed rule catches: 17 / 30000 = 0.0567%
Of those, declining: 16 / 17

Correlation of numeric features with is_declining:


days_with_impressions     0.190055
content_age_days          0.163882
age_tier_order            0.156142
trend_pct                 0.141068
impressions_last_30d      0.093980
word_count                0.090157
days_since_last_update    0.081383
char_count                0.072188
clicks_last_30d           0.071935
sessions_last_30d         0.063842
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.